# MFCC Extraction — Aymen's Genres
**Genres: Rock, Metal, Electronic, House/Dance, Country/Folk, Reggae**

Run this notebook fully. It will save results to `mfcc_aymen.csv`.
Cindy runs `MFCC_Cindy.ipynb` for her 6 genres at the same time.
Then merge both CSVs to get the full dataset.

In [ ]:
import os
import yt_dlp
import librosa
import numpy as np
import pandas as pd
from typing import Optional

# ── Aymen's 6 genres ──────────────────────────────────────────────────────────
MY_GENRES = ['Rock', 'Metal', 'Electronic', 'House/Dance', 'Country/Folk', 'Reggae']
OUTPUT_FILE = 'mfcc_aymen.csv'
TRACKS_PER_GENRE = 400  # ~2,400 total — adjust down if short on time

print(f'Extracting MFCCs for: {MY_GENRES}')
print(f'Target: {TRACKS_PER_GENRE} tracks per genre = {TRACKS_PER_GENRE * len(MY_GENRES)} total')

In [ ]:
GENRE_MAP = {
    'rock': 'Rock', 'alt-rock': 'Rock', 'alternative': 'Rock', 'indie': 'Rock',
    'grunge': 'Rock', 'punk': 'Rock', 'punk-rock': 'Rock', 'psych-rock': 'Rock',
    'rockabilly': 'Rock', 'rock-n-roll': 'Rock', 'hard-rock': 'Rock', 'emo': 'Rock',
    'metal': 'Metal', 'black-metal': 'Metal', 'death-metal': 'Metal',
    'heavy-metal': 'Metal', 'metalcore': 'Metal', 'grindcore': 'Metal',
    'electronic': 'Electronic', 'electro': 'Electronic', 'ambient': 'Electronic',
    'idm': 'Electronic', 'industrial': 'Electronic', 'new-age': 'Electronic', 'synth-pop': 'Electronic',
    'house': 'House/Dance', 'techno': 'House/Dance', 'trance': 'House/Dance',
    'dance': 'House/Dance', 'edm': 'House/Dance', 'dubstep': 'House/Dance',
    'drum-and-bass': 'House/Dance', 'garage': 'House/Dance',
    'country': 'Country/Folk', 'folk': 'Country/Folk', 'bluegrass': 'Country/Folk', 'honky-tonk': 'Country/Folk',
    'reggae': 'Reggae', 'dub': 'Reggae', 'ska': 'Reggae',
}

df = pd.read_csv('../Data/spotify-tracks-dataset.csv')
df = df.drop(columns=[c for c in ['Unnamed: 0.1', 'Unnamed: 0'] if c in df.columns])
df['genre'] = df['track_genre'].map(GENRE_MAP)
df = df.dropna(subset=['genre'])

# Filter to only Aymen's genres
df = df[df['genre'].isin(MY_GENRES)]

# Sample evenly — skip tracks already in existing mfcc_features.csv
existing = pd.read_csv('mfcc_features.csv') if os.path.exists('mfcc_features.csv') else pd.DataFrame(columns=['track_id'])
df = df[~df['track_id'].isin(existing['track_id'])]

sampled = df.groupby('genre', group_keys=False).apply(
    lambda g: g.sample(n=min(TRACKS_PER_GENRE, len(g)), random_state=42)
).reset_index(drop=True)

print(f'Tracks to process: {len(sampled)}')
print(sampled['genre'].value_counts())

In [ ]:
def get_audio_clip(track_name: str, artist: str, duration_ms: int = None,
                   out_dir: str = 'audio_clips_aymen') -> Optional[str]:
    os.makedirs(out_dir, exist_ok=True)
    query = f"{track_name} {artist} official audio"
    safe_name = f"{track_name[:40]}_{artist[:20]}".replace('/', '_').replace(' ', '_')
    out_path = os.path.join(out_dir, safe_name)

    if duration_ms:
        mid   = (duration_ms / 1000) // 2
        start = int(mid - 15)
        end   = int(mid + 15)
    else:
        start, end = 60, 90

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_path,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
        'quiet': True,
        'no_warnings': True,
        'default_search': 'ytsearch1',
        'download_ranges': lambda info, *_: [{'start_time': start, 'end_time': end}],
        'force_keyframes_at_cuts': True,
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([f'ytsearch1:{query}'])
        wav = out_path + '.wav'
        return wav if os.path.exists(wav) else None
    except Exception as e:
        return None


def extract_mfcc(filepath: str, n_mfcc: int = 13) -> Optional[np.ndarray]:
    try:
        y, sr = librosa.load(filepath, sr=22050, mono=True)
        mfccs  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
        deltas = librosa.feature.delta(mfccs)
        return np.concatenate([mfccs.mean(axis=1), mfccs.std(axis=1), deltas.mean(axis=1)])
    except Exception:
        return None

print('Functions ready.')

In [ ]:
n_mfcc = 13
feature_cols = (
    [f'mfcc_mean_{i+1}'  for i in range(n_mfcc)] +
    [f'mfcc_std_{i+1}'   for i in range(n_mfcc)] +
    [f'mfcc_delta_{i+1}' for i in range(n_mfcc)]
)

records = []
total = len(sampled)

for i, (_, row) in enumerate(sampled.iterrows()):
    print(f"[{i+1}/{total}] {row['genre']} | {row['track_name']} — {row['artists']}")

    filepath = get_audio_clip(row['track_name'], row['artists'], duration_ms=row['duration_ms'])
    if not filepath or not os.path.exists(filepath):
        print('  SKIP — download failed')
        continue

    features = extract_mfcc(filepath)
    if features is None:
        print('  SKIP — MFCC extraction failed')
        continue

    record = {'track_id': row['track_id'], 'track_genre': row['track_genre'], 'genre': row['genre']}
    record.update(dict(zip(feature_cols, features)))
    records.append(record)

    # Save every 50 tracks so you don't lose progress if it crashes
    if len(records) % 50 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
        print(f'  ✓ Checkpoint saved — {len(records)} tracks so far')

    try:
        os.remove(filepath)
    except Exception:
        pass

result_df = pd.DataFrame(records)
result_df.to_csv(OUTPUT_FILE, index=False)
print(f'\nDone! {len(result_df)} tracks saved to {OUTPUT_FILE}')
print(result_df['genre'].value_counts())

In [ ]:
# ── Merge with Cindy's output when both are done ──────────────────────────────
# Run this cell ONLY after both mfcc_aymen.csv and mfcc_cindy.csv exist

if os.path.exists('mfcc_aymen.csv') and os.path.exists('mfcc_cindy.csv'):
    aymen = pd.read_csv('mfcc_aymen.csv')
    cindy = pd.read_csv('mfcc_cindy.csv')
    combined = pd.concat([aymen, cindy], ignore_index=True)
    combined = combined.drop_duplicates(subset='track_id')
    combined.to_csv('mfcc_features_v2.csv', index=False)
    print(f'Merged: {len(combined)} total tracks saved to mfcc_features_v2.csv')
    print(combined['genre'].value_counts())
else:
    print('Waiting for both files — run both notebooks first.')